# 04 Embedding Baseline

Purpose: embed degree module texts and cleaned job texts, then compute the baseline degree-job similarity matrix.

Inputs:
- `data/processed/degree_modules.parquet`
- `data/processed/degree_profiles.parquet`
- `data/interim/jobs_clean.parquet`

Outputs:
- `data/embeddings/degree_module_embeddings_all-MiniLM-L6-v2.npy`
- `data/embeddings/degree_embeddings_all-MiniLM-L6-v2.npy`
- `data/embeddings/job_embeddings_all-MiniLM-L6-v2.npy`
- `data/embeddings/similarity_matrix_all-MiniLM-L6-v2.npy`
- `data/figures/similarity_distributions.png`


In [1]:
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings('ignore')
repo_root = Path.cwd().resolve()
if repo_root.name == 'notebooks':
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from analysis.io import paths, require_files

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

from analysis.embedding import build_degree_embeddings, compute_similarity_matrix, load_or_compute_embeddings, top_k_jobs_for_degree
from analysis.plotting import plot_similarity_distributions
from analysis.preprocessing import build_structured_job_text
from analysis.profiles import build_degree_module_indices, build_module_text

MODEL_NAME = 'all-MiniLM-L6-v2'
DEGREE_MODULE_TOP_K = 5

require_files([paths.degree_modules_parquet, paths.degree_profiles_parquet, paths.jobs_clean_parquet])
print('Expected upstream inputs: cleaned jobs and processed degree artifacts')


Expected upstream inputs: cleaned jobs and processed degree artifacts


In [2]:
degree_modules = pd.read_parquet(paths.degree_modules_parquet)
degree_profiles = pd.read_parquet(paths.degree_profiles_parquet)
jobs = pd.read_parquet(paths.jobs_clean_parquet)

if 'module_profile_text' not in degree_modules.columns:
    degree_modules['module_profile_text'] = degree_modules.apply(build_module_text, axis=1)
if 'job_text' not in jobs.columns:
    jobs['job_text'] = jobs.apply(build_structured_job_text, axis=1)

degree_module_indices = build_degree_module_indices(degree_profiles, degree_modules)
module_embeddings = load_or_compute_embeddings(
    degree_modules['module_profile_text'].tolist(),
    paths.degree_module_embeddings_npy(MODEL_NAME),
    model_name=MODEL_NAME,
    batch_size=32,
)

if paths.degree_embeddings_npy(MODEL_NAME).exists():
    degree_embeddings = np.load(paths.degree_embeddings_npy(MODEL_NAME))
    if degree_embeddings.shape[0] != len(degree_profiles):
        degree_embeddings = build_degree_embeddings(degree_profiles, degree_module_indices, module_embeddings)
        np.save(paths.degree_embeddings_npy(MODEL_NAME), degree_embeddings)
else:
    degree_embeddings = build_degree_embeddings(degree_profiles, degree_module_indices, module_embeddings)
    np.save(paths.degree_embeddings_npy(MODEL_NAME), degree_embeddings)

job_embeddings = load_or_compute_embeddings(
    jobs['job_text'].tolist(),
    paths.job_embeddings_npy(MODEL_NAME),
    model_name=MODEL_NAME,
    batch_size=64,
)

if paths.similarity_matrix_npy(MODEL_NAME).exists():
    sim_matrix = np.load(paths.similarity_matrix_npy(MODEL_NAME))
    expected_shape = (len(degree_profiles), len(jobs))
    if sim_matrix.shape != expected_shape:
        sim_matrix = compute_similarity_matrix(degree_profiles, degree_module_indices, module_embeddings, job_embeddings, DEGREE_MODULE_TOP_K)
        np.save(paths.similarity_matrix_npy(MODEL_NAME), sim_matrix)
else:
    sim_matrix = compute_similarity_matrix(degree_profiles, degree_module_indices, module_embeddings, job_embeddings, DEGREE_MODULE_TOP_K)
    np.save(paths.similarity_matrix_npy(MODEL_NAME), sim_matrix)

print(f'Module embeddings shape: {module_embeddings.shape}')
print(f'Degree embeddings shape: {degree_embeddings.shape}')
print(f'Job embeddings shape: {job_embeddings.shape}')
print(f'Similarity matrix shape: {sim_matrix.shape}')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11733.75it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:  25%|██▌       | 1/4 [00:00<00:01,  2.53it/s]

...

Batches:  99%|█████████▉| 269/272 [02:17<00:01,  2.15it/s]

Batches:  99%|█████████▉| 270/272 [02:18<00:00,  2.38it/s]

Batches: 100%|█████████▉| 271/272 [02:18<00:00,  2.26it/s]

Batches: 100%|██████████| 272/272 [02:18<00:00,  1.96it/s]

Module embeddings shape: (115, 384)
Degree embeddings shape: (5, 384)
Job embeddings shape: (17346, 384)
Similarity matrix shape: (5, 17346)


In [3]:
for degree_id in ['cs', 'dsa']:
    if (degree_profiles['degree_id'] == degree_id).any():
        print(f'Top-10 jobs for {degree_id}:')
        display(top_k_jobs_for_degree(sim_matrix, degree_profiles, jobs, degree_id, 'sim_score', k=10))

plot_similarity_distributions(sim_matrix, degree_profiles, paths.similarity_distribution_png)
print(f'Saved similarity distribution figure to: {paths.similarity_distribution_png}')


Top-10 jobs for cs:


,rank,job_id,job_title,company,categories,sim_score
0,1,MCF-2025-1715506,Software Development Engineer,NUTEK PRIVATE LIMITED,"Engineering, Manufacturing",0.4751
1,2,MCF-2025-1801982,Software Engineer (Precision / Automation),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4723
2,3,MCF-2025-1801975,Software Engineer (Semiconductor Industry),WECRUIT PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4700
3,4,MCF-2026-0190683,Software Application Engineer - WCAN,WECRUIT PTE. LTD.,Information Technology,0.4629
4,5,MCF-2026-0188112,"Software Senior /Engineer (C#,PLC)- Automated ...",PEOPLE PROFILERS PTE. LTD.,"Engineering, Sciences / Laboratory / R&D",0.4602
5,6,MCF-2026-0189542,Software Developer,ALFA CONNECTIONS PTE. LTD.,Information Technology,0.4502
6,7,MCF-2026-0170526,"Software Engineer (C++/C#, Machine Automation)...",ANRADUS PTE. LTD.,"Information Technology, Others",0.4487
7,8,MCF-2026-0159945,Junior Software Engineer,ALPHA X TECHNOLOGY PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.4460
8,9,MCF-2026-0167383,Applications Developer (Cards Technology),A-IT SOFTWARE SERVICES PTE LTD,"Banking and Finance, Information Technology",0.4456
9,10,MCF-2026-0190672,System Application Engineer - WCAN,WECRUIT PTE. LTD.,Information Technology,0.4452


Top-10 jobs for dsa:


,rank,job_id,job_title,company,categories,sim_score
0,1,MCF-2025-1604902,Data Analyst,MSI GLOBAL PRIVATE LIMITED,"Engineering, Information Technology, Others",0.5470
1,2,MCF-2026-0186942,Principal Data Scientist (Aerospace),RN CARE PTE. LTD.,"Engineering, Information Technology, Manufactu...",0.5441
2,3,MCF-2025-1424769,Data Engineer,DEEGIT ASIA PTE. LTD.,Information Technology,0.5233
3,4,MCF-2025-1164543,Outreach Data Analyst,PERSOL SINGAPORE PTE. LTD.,Public / Civil Service,0.5174
4,5,MCF-2026-0161241,Senior Data Analyst (Bank | Up to 9.5k),ADECCO PERSONNEL PTE LTD,Banking and Finance,0.5087
5,6,MCF-2025-1869712,Data Analyst Lead,INNOVATIQ TECHNOLOGIES PTE. LTD.,"Information Technology, Sciences / Laboratory ...",0.5069
6,7,MCF-2026-0176414,Data Engineer / Data Analyst,L&M FOUNDATION SPECIALIST PTE LTD,Building and Construction,0.5069
7,8,MCF-2026-0183284,Data Analyst,IOTALENTS PTE. LTD.,Information Technology,0.5024
8,9,MCF-2026-0160469,Data Analyst,APAR TECHNOLOGIES PTE. LTD.,Information Technology,0.5020
9,10,MCF-2026-0175746,Data Scientist,QUANTEAM (SINGAPORE) PTE. LTD.,Consulting,0.4999


Saved similarity distribution figure to: /Users/marcusyeo/Github/DSA4264-Text-Group-3/data/figures/similarity_distributions.png
